In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu numpy requests python-dotenv streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 20.5 MB/s eta 0:00:00


In [ ]:
import pypdf
import sentence_transformers
import faiss
import numpy as np
import requests
import dotenv
import streamlit

print("pypdf:", pypdf.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss: OK, version attr not always present")
print("numpy:", np.__version__)
print("streamlit:", streamlit.__version__)
print("All imports successful ✅")

pypdf: 6.19.0
sentence-transformers: 5.7.0
faiss: OK, version attr not always present
numpy: 2.1.3
streamlit: 1.64.0
All imports successful ✅


In [ ]:
%%writefile app.py
import os
import io
import requests
import numpy as np
import faiss
import streamlit as st
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
TOP_K = 3

# ---------- Step 1: PDF Extraction ----------
def extract_pdf_text(file_bytes):
    reader = PdfReader(io.BytesIO(file_bytes))
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages

# ---------- Step 2: Chunking ----------
def chunk_text(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    for p in pages:
        text = p["text"]
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            if chunk.strip():
                chunks.append({"page": p["page"], "text": chunk})
            start += chunk_size - overlap
    return chunks

# ---------- Step 3: Embeddings ----------
@st.cache_resource
def load_embedding_model():
    return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def create_embeddings(chunks, model):
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    faiss.normalize_L2(embeddings)
    return embeddings

# ---------- Step 4: FAISS ----------
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

def retrieve_relevant_chunks(question, model, index, chunks, top_k=TOP_K):
    q_emb = model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "score": float(score)
        })
    return results

# ---------- Step 5: OpenRouter LLM ----------
def generate_answer(question, context_chunks):
    if not OPENROUTER_API_KEY:
        return "⚠️ Configuration error: OPENROUTER_API_KEY is missing. Please set it in your .env file."

    context = "\n\n".join(
        [f"[Page {c['page']}]: {c['text']}" for c in context_chunks]
    )

    prompt = f"""You are an AI document assistant.

Answer the user's question using ONLY the provided document context.

Do not invent facts.

If the answer cannot be found in the provided context, say:
"I couldn't find that information in the uploaded document."

Keep the answer concise and clear.

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}
"""

    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": OPENROUTER_MODEL,
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"].strip()
    except requests.exceptions.RequestException as e:
        return f"⚠️ Error contacting OpenRouter: {e}"
    except (KeyError, IndexError):
        return "⚠️ Unexpected response format from OpenRouter."

# ---------- Streamlit UI ----------
st.set_page_config(page_title="AI Document Q&A", page_icon="📄", layout="wide")

st.title("📄 AI Document Q&A")
st.caption("Ask questions about your document using Retrieval-Augmented Generation")

if "chunks" not in st.session_state:
    st.session_state.chunks = None
if "index" not in st.session_state:
    st.session_state.index = None
if "filename" not in st.session_state:
    st.session_state.filename = None
if "num_pages" not in st.session_state:
    st.session_state.num_pages = 0

model = load_embedding_model()

with st.sidebar:
    st.header("📁 Document")
    uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])

    if uploaded_file is not None:
        if st.session_state.filename != uploaded_file.name:
            with st.spinner("Processing document..."):
                try:
                    file_bytes = uploaded_file.read()
                    pages = extract_pdf_text(file_bytes)

                    if not pages:
                        st.error("❌ No readable text found. This may be a scanned/image-only PDF.")
                    else:
                        chunks = chunk_text(pages)
                        embeddings = create_embeddings(chunks, model)
                        index = build_faiss_index(embeddings)

                        st.session_state.chunks = chunks
                        st.session_state.index = index
                        st.session_state.filename = uploaded_file.name
                        st.session_state.num_pages = len(pages)
                        st.success("✅ Document processed successfully")
                except Exception as e:
                    st.error(f"❌ Failed to process PDF: {e}")

    if st.session_state.filename:
        st.markdown("---")
        st.write(f"**File:** {st.session_state.filename}")
        st.write(f"**Pages:** {st.session_state.num_pages}")
        st.write(f"**Chunks:** {len(st.session_state.chunks)}")

st.subheader("Ask a question about your document")
question = st.text_input("Your question", placeholder="e.g. What is the main conclusion of this document?")
ask_clicked = st.button("Ask AI")

if ask_clicked:
    if not st.session_state.chunks:
        st.warning("⚠️ Please upload a PDF first.")
    elif not question.strip():
        st.warning("⚠️ Please enter a question.")
    else:
        with st.spinner("Searching document and generating answer..."):
            try:
                results = retrieve_relevant_chunks(
                    question, model, st.session_state.index, st.session_state.chunks
                )
                if not results:
                    st.warning("No relevant content found in the document.")
                else:
                    answer = generate_answer(question, results)

                    st.markdown("### 🤖 Answer")
                    st.write(answer)

                    st.markdown("### 📚 Sources")
                    for r in results:
                        with st.expander(f"Page {r['page']} — similarity score {r['score']:.3f}"):
                            st.write(r["text"])
            except Exception as e:
                st.error(f"❌ Unexpected error: {e}")

Overwriting app.py


In [ ]:
!pip install -q streamlit pypdf sentence-transformers faiss-cpu numpy requests python-dotenv pyngrok

In [ ]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_MODEL"] = "openai/gpt-4o-mini"

Enter your OpenRouter API key: ··········


In [ ]:
import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Streamlit started.")

Streamlit started.


In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Fri, 25 Sep 2026 07:55:50 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Fri, 25 Sep 2026 07:27:12 GMT
etag: "ab0ee3ab3e931a1b0c1eb952db2828b5"
cache-control: no-cache



In [ ]:
from pyngrok import ngrok
from getpass import getpass

ngrok_auth_token = getpass("Enter your ngrok Auth Token: ")
ngrok.set_auth_token(ngrok_auth_token)

public_url = ngrok.connect(8501)

print("App URL:", public_url)

Enter your ngrok Auth Token: ··········
App URL: NgrokTunnel: "https://b4f4-34-139-52-195.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
%%writefile requirements.txt
streamlit
pypdf
sentence-transformers
faiss-cpu
numpy
requests
python-dotenv

Writing requirements.txt


In [ ]:
%%writefile .env.example
OPENROUTER_API_KEY=
OPENROUTER_MODEL=openai/gpt-4o-mini

Writing .env.example


In [ ]:
%%writefile .gitignore
.env
__pycache__/
*.pyc
.venv/
venv/

Overwriting .gitignore


In [ ]:
import os
from getpass import getpass
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_MODEL"] = "openai/gpt-4o-mini"

Enter your OpenRouter API key: ··········


In [ ]:
import os
!pip install -q pyngrok

from pyngrok import ngrok
from getpass import getpass

ngrok_auth_token = getpass("Enter your ngrok Auth Token: ")

ngrok.set_auth_token(ngrok_auth_token)

get_ipython().system_raw("streamlit run app.py &")

public_url = ngrok.connect(8501)
print("App URL:", public_url)

Enter your ngrok Auth Token: ··········
App URL: NgrokTunnel: "https://be5e-34-139-52-195.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501



2026-09-25 07:47:37.052 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.52.195:8501

  Stopping...
